# Week 2 / Day 01
## Covered today
1. Connecting to Multiple Frontier Models with APIs (OpenAI, Claude, Gemini)
2. Testing GPT-5 Models with Reasoning Effort and Scaling Puzzles
3. Testing Claude, GPT-5, Gemini & DeepSeek on Brain Teasers
4. Local Models with Ollama, Native APIs, and OpenRouter Integration
5. LangChain vs LiteLLM: Choosing the Right LLM Framework
6. LLM vs LLM: Building Multi-Model Conversations with OpenAI & Claude

### Now we make all the required imports

In [1]:
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display


### Now we load all the API Keys

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

### Now check if all loaded keys exist and are as per the format

In [3]:
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")


if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")


if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")


if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [4]:
# create clients for each provider
# for openai, we simply use OpenAI, for others we need to specify the base url and key, while using openai api library
openai_client = OpenAI()

google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'

In [5]:
google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

In [6]:
# Now test if each is working
tell_a_joke = [
    {'role': 'user', 'content': 'Tell a joke for a student who is on a journey to become an LLM engineer.'}
]
response = openai_client.chat.completions.create(model='gpt-5-mini', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

Sure — here’s one tailored to you:

Why did the aspiring LLM engineer bring a ladder to the lab?  
Because their model kept asking to "scale up" — so they took it literally.

Want a couple more short ones?


In [7]:
response = anthropic_client.chat.completions.create(model='claude-haiku-4-5-20251001', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

# Why did the LLM engineer go to therapy?

Because they had too many **layers** of emotional baggage, and no amount of **attention** could help them focus! 

Plus, they kept trying to **fine-tune** their life, but it just kept **hallucinating** better versions of themselves. 🤖

---

*Bonus wisdom for your journey: May your gradients always flow, your losses always decrease, and your GPU memory never run out!* 💻✨


In [8]:
response = google_client.chat.completions.create(model='gemini-3.5-flash-lite', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

Why did the aspiring LLM Engineer break up with their romantic partner?

Because every time they tried to have a normal conversation, the partner felt the need to **fine-tune** it, accused them of **hallucinating**, and kept asking for their **API key** just to feel validated. 

Plus, the relationship lacked **attention**—it was all self-attention, all the time.


In [9]:
response = ollama_client.chat.completions.create(model='gemma3:270m', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

Why did the AI fail its assignment? 

Because it couldn't remember where it came from!



In [10]:
response = openrouter_client.chat.completions.create(model='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

Why did the aspiring LLM engineer bring a ladder to the neural‑network party?

Because they heard the model was *deep* and wanted to reach the “top‑level” API! 🚀😄


### Training vs Inference Scaling

Next up, we will have an easy puzzle and we will send this as a prompt to small models, with reasoning. We shall increase the reasoning and also add a bigger model to check if the answers are correct and at what level.

In [11]:
easy_puzzle = [
    {'role': 'user', 'content': 
    "You toss 2 coins. One of them is heads. What's the probablity that the other is tails. Answer with probablity only."}
]

In [12]:
response = openai_client.chat.completions.create(model='gpt-5-nano', messages=easy_puzzle, reasoning_effort='minimal')
print(response.choices[0].message.content)

1/2


This above is a wrong answer. Now, we increase the reasoning to low.

In [13]:
response = openai_client.chat.completions.create(model='gpt-5-nano', messages=easy_puzzle, reasoning_effort='low')
print(response.choices[0].message.content)

2/3


Now, we can see that with increase in reasoning, the answer becomes correct. Now, with minimal reasoning, we bump up the model from nano to mini

In [14]:
response = openai_client.chat.completions.create(model='gpt-5-mini', messages=easy_puzzle, reasoning_effort='minimal')
print(response.choices[0].message.content)

2/3


With now, even the minimal reasoning, but a bumped up model, correct answer was received.

### Nextup we work with Anthropic and Google Client Libraries

We start with google, by first installing -U google-genai and then importing genai from google

In [15]:
from google import genai

In [16]:
client = genai.Client()
response = client.interactions.create(
    model='gemini-3.5-flash-lite',
    input='Describe color blue who has never been able to see.'
)
print(response.output_text)

How do you explain a color to someone who has never known sight? You cannot use the eyes; instead, you must translate light into the language of the other senses—into feeling, sound, and temperature. 

If you have never seen blue, I would ask you to imagine it not as a sight, but as an experience. 

Think about the air on the very first warm day of spring, just as the frost begins to melt. Take a deep, slow breath through your nose. That coolness filling your chest—crisp, clean, and deep—that is the essence of blue. It is the feeling of breathing in wide-open space. 

Now, think about touch. Imagine walking barefoot across smooth, cool river stones in the shade of a great tree, while a gentle breeze rolls over your skin. Blue is the sensation of that coolness. It is the opposite of a sharp, stinging heat. It is gentle, steady, and calming. 

If blue had a sound, it would not be a sudden, sharp crash, nor would it be a high, piercing shriek. It would be a deep, low cello note sustained 

Next we move to anthropic, first we install anthropic, then we from anthropic, we import Anthropic

In [ ]:
from anthropic import Anthropic
client = Anthropic()
response = client.messages.create(
    max_tokens=1000,
    model='claude-haiku-4-5-20251001',
    messages=[
        {'role': 'user', 'content':'Describe color blue who has never been able to see.'}
    ]
)
print(response.content[0].text)

# Blue for Someone Who Has Never Seen

Imagine the feeling of **cool water** on your skin—that's close to what blue feels like. It's a temperature, almost.

**Blue is calm.** Think of how your body feels in a quiet, safe place—that gentle ease. Blue doesn't excite or demand attention. It lets you rest.

**In sound:** Blue might be a low note on a cello or the hum of a refrigerator—steady, dependable, not jarring.

**In taste:** Some say blue tastes like cool mint or clean water—refreshing, clear, with no sharp edges.

**In texture:** Imagine smooth glass or still water—surfaces that are cool to touch and undisturbed. Nothing rough about it.

**In emotion:** Blue is often sad, yes—like when you feel a quiet grief or longing. But it's also peaceful, like the emotion of acceptance or contemplation. It's the feeling of an empty sky when you're thinking deeply.

**In nature:** Rain, deep ocean, shadows under trees at dusk, the air on a clear night—all that coolness, all that calm openness.


While Open AI SDK is used mostly, since they were the first to start the AI revolution and most people have moved from Open AI SDK, sometimes people can use SDKs of the providers they are using, hence, it is good to understand how their SDKs work.

### Next we move to Abstraction Layers (Langchain & LiteLLM)

This is just a very very brief intro to langchain, only the syntax part and a brief explanation of abstraction layers, so we simply code along. we start by installing langchain_openai

In [20]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-5-mini')
response = llm.invoke(tell_a_joke)
print(response.content)

1) Why did the aspiring LLM engineer bring a ladder to class? They heard they needed to work on their model's layers.

2) How does an LLM engineer go on a spiritual journey? They do gradient descent until they find inner minima.

3) Why did the student fine-tune their relationship? They wanted fewer hallucinations and better context retention.

4) I told my model a joke — it finished the punchline with 0.02 confidence. Guess we both need more training.

Want one tailored to a specific part of the journey (prompting, debugging, research papers, etc.)?


### Moving on to LiteLLM

In [ ]:
# like above we only look at the syntax for now, later we deep dive
from litellm import completion
response = completion(model='openai/gpt-4.1', messages=tell_a_joke)
print(response.choices[0].message.content)

Why did the student bring a transformer to the study group?

Because they heard it could help them with their attention!


In addition to providing a simple interface to run multiple models/providers, litellm also gives a very simply interface to display, input tokens, output tokens, total tokens and cost involved. We shall see it for the last LLM request below

In [35]:
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens: {response.usage.total_tokens}")
print(f"Cached Tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Input Tokens: 25
Output Tokens: 23
Total Tokens: 48
Cached Tokens: 0
Total Cost: 0.023 cents


### Next we move to prompt caching

We check the token caching using litellm

In [36]:
with open("hamlet.txt", "r", encoding="utf-8") as file:
    hamlet = file.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [49]:
question = [
    {'role': 'user', 'content': "In hamlet, when Laertes asks 'Where is my father?' what is the reply?"}
]

In [50]:
response = completion(model='gemini/gemini-3.5-flash-lite', messages=question)
print(response.choices[0].message.content)

When Laertes bursts into the palace demanding to know where his father is (in Act 4, Scene 5), King Claudius replies:

**"Dead."**

Queen Gertrude immediately tries to soften the blow and deflect the blame, adding: 

**"But not by him."** (meaning Claudius didn't do it).


In [51]:
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens: {response.usage.total_tokens}")
print(f"Cached Tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Input Tokens: 19
Output Tokens: 71
Total Tokens: 90
Cached Tokens: None
Total Cost: 0.018 cents


In [45]:
response = completion(model='gemini/gemini-3.5-flash-lite', messages=question)
print(response.choices[0].message.content)
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens: {response.usage.total_tokens}")
print(f"Cached Tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Speak, man. The reply is **"Dead."** (Spoken by the King).
Input Tokens: 64
Output Tokens: 19
Total Tokens: 83
Cached Tokens: None
Total Cost: 0.007 cents


In [53]:
question[0]['content'] += "\n\nFor context, here is the entire text of Hamlet\n" + hamlet


In [54]:
response = completion(model='gemini/gemini-3.5-flash-lite', messages=question)
print(response.choices[0].message.content)
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens: {response.usage.total_tokens}")
print(f"Cached Tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Based on the text of Shakespeare's *Hamlet*, when Laertes breaks in and demands "Where is my father?", the reply is given by the King:

> **King.** Dead.
> 
> **Queen.** But not by him!
> 
> **King.** Let him demand his fill.
Input Tokens: 53207
Output Tokens: 66
Total Tokens: 53273
Cached Tokens: None
Total Cost: 1.613 cents


In [55]:
response = completion(model='gemini/gemini-3.5-flash-lite', messages=question)
print(response.choices[0].message.content)
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens: {response.usage.total_tokens}")
print(f"Cached Tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Based on the text provided from Act IV, Scene V of *Hamlet*, when Laertes asks "Where is my father?", the King's reply is:

> **"Dead."**
Input Tokens: 53207
Output Tokens: 39
Total Tokens: 53246
Cached Tokens: 49122
Total Cost: 0.280 cents


Now if we see this again, we ran the same command, now, in this case 49k or so tokens have been cached and if we look at the total cost, from 1.6 to 0.2 cents, due to cached tokens. Now, for anthropic, gemini and openai, the rules for caching are different, this is something we need to review before we define a program and want to apply caching based on the LLM provider.